<a href="https://colab.research.google.com/github/auzaluis/upsa_mod_202601/blob/main/personalidad/01_script_ETL_personalidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tema 01: Carga de datos

## Importando base de datos

In [ ]:
# Google Auth
from google.colab import auth
auth.authenticate_user()

In [ ]:
# API Client
from google.auth import default
creds, _ = default()

In [ ]:
# gspread authorization
import gspread
gc = gspread.authorize(creds)

In [ ]:
# Accediendo al Google Sheet
url = 'https://docs.google.com/spreadsheets/d/1IQ_RxxTSmBKHTExlxboIRNlMov_F6RyqdcOPrflCv_w/edit?usp=sharing'
gsheets = gc.open_by_url(url)
sheets = gsheets.worksheet('Respuestas de formulario 1').get_all_values()

In [ ]:
type(sheets)

In [ ]:
# Convirtiendo a data frame
import pandas as pd
df = pd.DataFrame(sheets[1:], columns=sheets[0])

## Inspección del data frame

In [ ]:
type(df)

In [ ]:
df.head()

In [ ]:
# Analizar la estructura de un df
df.info()

In [ ]:
# Tupla con la cantidad de filas y columnas
df.shape

In [ ]:
# Cantidad de filas
len(df)

In [ ]:
# Cantidad de columnas
len(df.columns)

In [ ]:
df.columns

## Tema 02: Transformación de datos

### Valores perdidos

Identificando NAs

In [ ]:
df[['Escribe tu edad exacta']] # Los NAs están como strings vacíos

In [ ]:
import numpy as np
df.replace('', np.nan, inplace=True)

In [ ]:
df.info()

In [ ]:
(
    df['Escribe tu edad exacta']
    .isna()
    .value_counts()
)

In [ ]:
df['Escribe tu edad exacta'].dtype

In [ ]:
df['Escribe tu edad exacta'] = pd.to_numeric(df['Escribe tu edad exacta'], errors='coerce')

In [ ]:
df['Escribe tu edad exacta'].dtype

Imputación por el promedio

In [ ]:
# Calcular la media
edad_promedio = df['Escribe tu edad exacta'].mean().round(0)
edad_promedio

In [ ]:
# Creando un nuevo df
df2 = df.copy()

In [ ]:
# Reemplazo por la media
df2['edad2'] = df2['Escribe tu edad exacta'].fillna(edad_promedio)

In [ ]:
df2[['Escribe tu edad exacta', 'edad2']]

In [ ]:
df2.info()

In [ ]:
# Relocalizar la columna 'edad2' después de 'Escribe tu edad exacta'
# new_index = list(df2.columns).index('Escribe tu edad exacta') + 1
# df2.insert(new_index, 'edad2', df2.pop('edad2'))

Creando función relocate en python

In [ ]:
def relocate(df, columna, after):
  new_index = list(df.columns).index(after) + 1
  df.insert(new_index, columna, df.pop(columna))
  return df

In [ ]:
relocate(df=df2, columna='edad2', after='Escribe tu edad exacta')

In [ ]:
df2.info()

Eliminando filas que contienen NAs

In [ ]:
df2.shape

In [ ]:
df2.dropna(inplace=True)

In [ ]:
df2.shape

### Valores perdidos

Normalización

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
# Instanciando StandardScaler()
normalizador = StandardScaler()

In [ ]:
df2['edad2'].head()

In [ ]:
# normalizando
normalizador.fit_transform(df2[['edad2']])

In [ ]:
df3 = df2.copy()

In [ ]:
df3['edadZ'] = normalizador.fit_transform(df3[['edad2']])

In [39]:
df3 = relocate(df=df3, columna='edadZ', after='edad2')

Rango (min=0 - max=1)

In [41]:
from sklearn.preprocessing import MinMaxScaler

In [42]:
# Instanciando minmax
rango = MinMaxScaler()

In [43]:
df3['edad_rango'] = rango.fit_transform(df3[['edad2']])

In [44]:
df3 = relocate(df=df3, columna='edad_rango', after='edadZ')

In [ ]:
df3

### Agrupaciones

Numéricas

In [47]:
cortes = [-float('inf'), 18, 21, float('inf')]
etiquetas = ['18 o menos', '19 a 21', 'Más de 21']

In [48]:
df3['edadGR'] = pd.cut(df3['edad2'], bins=cortes, labels=etiquetas)

In [50]:
df3 = relocate(df=df3, columna='edadGR', after='edad_rango')

In [ ]:
df3.value_counts('edadGR')

In [ ]:
df3[['edad2', 'edadZ', 'edad_rango', 'edadGR']].head()

Categóricas

In [ ]:
df3.info()

In [ ]:
df3.iloc[:,8]

In [ ]:
df3.iloc[:,8].value_counts()

In [58]:
df3.iloc[:,8].isin(['Totalmente verdadero', 'Un poco verdadero']).sum()

np.int64(219)

In [59]:
# Función top2box
def top2box(x):
  if x in ['Totalmente verdadero', 'Un poco verdadero']:
    return 1
  else:
    return 0

In [61]:
df3.iloc[:,8].apply(top2box).sum()

np.int64(219)

In [62]:
# Función lambda
df3.iloc[:,8].apply(
    lambda x: 1 if x in ['Totalmente verdadero', 'Un poco verdadero'] else 0
).sum()

np.int64(219)

**Bucles**

Manera standard

In [63]:
df4 = df3.copy()

In [65]:
# Crear una lista vacía
frases = []

# Bucle para llenar la lista
for col in df4.columns:
  if col.startswith('Según'):
    frases.append(col)

In [ ]:
frases

In [67]:
for frase in frases:
  df4[frase] = df4[frase].apply(top2box)

In [ ]:
df4[frases].head()